# RRR：基参数辨识

这个例子用三轴 RRR 机械臂说明基参数辨识：先核对符号与数值回归矩阵，再搜索激励轨迹、拟合基参数，最后由基参数计算 `M / c / g`。按顺序运行即可；建模和公式导出见 [modeling.ipynb](modeling.ipynb)。

数据由解析轨迹和 MuJoCo `mj_inverse` 生成，位置、速度和加速度都是已知值。加噪声一节只扰动力矩，没有闭环跟踪、编码器噪声或数值微分，因此这里的误差不能和 [README 的闭环实验](../../README.md) 直接比较。

在仓库根目录安装依赖：`pip install -e ".[sim,ident,viz,dev]"`。Notebook 可从仓库或自身所在目录启动。


In [ ]:
import sys
from pathlib import Path


def _project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "robot_model" / "__init__.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("找不到含 pyproject.toml 和 src/robot_model 的项目根目录")


_ROOT = _project_root(Path.cwd().resolve())
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import mujoco
import numpy as np
import sympy
from IPython.display import display

from robot_model import Robot
from robot_model.data import add_noise, sample_trajectory
from robot_model.excitation import search_excitation
from robot_model.identification import (
    DynamicsDecomposer,
    identify,
    parameter_error,
)
from robot_model.robots.rrr import RRR_BODY_NAMES, RRR_MDH_PARMS, rrr_xml_path
from robot_model.simulation.reference import MujocoReference

print("project root:", _ROOT)
print("xml:", rrr_xml_path())


## 1. 建模并核对正运动学

MDH 参数来自 `robot_model.robots.rrr`。`rrr_arm.xml` 提供本例的 MuJoCo 参考模型；它不含关节摩擦和电机惯量，适合检验纯刚体动力学。


In [ ]:
robot = Robot.from_mdh("rrr_arm", RRR_MDH_PARMS)
robot.print_dh_table()
print(f"dof={robot.dof}, convention={robot.convention}")

model = mujoco.MjModel.from_xml_path(str(rrr_xml_path()))
ref = MujocoReference(model, RRR_BODY_NAMES)

q_fk = np.array([0.3, -0.5, 0.8])
T_lib = robot.fk_numpy(q_fk)
T_mj = ref.body_poses(q_fk)[-1]
print("末端位置误差:", np.linalg.norm(T_lib[:3, 3] - T_mj[:3, 3]))
print("末端姿态误差:", np.linalg.norm(T_lib[:3, :3] - T_mj[:3, :3]))


## 2. 符号回归矩阵

RRR 只有三个关节，可以直接生成符号回归矩阵 `H`。QR 分解选出独立列后，得到基参数回归矩阵 `Hb = H Pb`，再用 `lambdify` 转成数值函数。


In [ ]:
dyn = robot.dynamics
dyn.gen_regressor()
dyn.calc_base_parms()
Hb_sym = dyn.gen_base_regressor()
print(f"完整参数 {dyn.n_dynparms} → 基参数 {dyn.n_base}")
print("前 3 个基参数表达式:")
for expr in list(dyn.baseparms)[:3]:
    display(sympy.expand(expr))

s = robot.symbols
Hb_lambdified = sympy.lambdify(
    list(s.q) + list(s.dq) + list(s.ddq), Hb_sym, "numpy"
)


def Hb_symbolic(q, dq, ddq):
    return np.asarray(Hb_lambdified(*(list(q) + list(dq) + list(ddq))), float)


pi_true = np.asarray(ref.dynparms(), float)
pi_b_true = np.asarray(
    sympy.lambdify(list(s.dynparms()), sympy.Matrix(dyn.baseparms), "numpy")(*pi_true),
    float,
).ravel()
print("pi_b_true shape:", pi_b_true.shape)


## 3. 数值回归矩阵

数值路径用递归牛顿–欧拉算法计算回归矩阵，不展开符号 `H`。下面另建一个模型，以免复用上一节的符号计算缓存。

两条路径使用相同的模型和基参数定义，在同一状态下比较 `Hb`，并分别检查 `Hb @ pi_b` 与 MuJoCo 力矩是否一致。


In [ ]:
# 重新建一个干净模型，避免复用上面已生成的符号 H
robot_num = Robot.from_mdh("rrr_arm_numeric", RRR_MDH_PARMS)
dyn_num = robot_num.dynamics
dyn_num.calc_base_parms(samples=400, numeric=True)  # 数值采样 QR
dyn_num.gen_base_regressor()
Hb_numeric = dyn_num.Hb_func

q0 = np.array([0.2, -0.4, 0.6])
dq0 = np.array([0.1, -0.2, 0.3])
ddq0 = np.array([-0.5, 0.4, -0.1])
Hs = Hb_symbolic(q0, dq0, ddq0)
Hn = Hb_numeric(q0, dq0, ddq0)
print("Hb symbolic shape:", Hs.shape, "numeric shape:", Hn.shape)
print("||Hs - Hn||_F =", np.linalg.norm(Hs - Hn))
print("||Hs @ pi_b - tau_mj|| =", np.linalg.norm(Hs @ pi_b_true - ref.inverse_dynamics(q0, dq0, ddq0)))
print("||Hn @ pi_b - tau_mj|| =", np.linalg.norm(Hn @ pi_b_true - ref.inverse_dynamics(q0, dq0, ddq0)))


## 4. 生成理想数据并辨识

默认使用上一节的数值回归矩阵。先搜索一条条件数较小的激励轨迹，再用 `mj_inverse` 计算力矩，最小二乘拟合基参数。

第一次拟合不加噪声，用来检查整条计算链是否一致。后面加入不同强度的力矩噪声，观察参数误差和拟合残差如何变化；噪声标准差的单位是 N·m。


In [ ]:
Hb_func = Hb_numeric  # 也可换成 Hb_symbolic

rng = np.random.default_rng(0)
traj, cond = search_excitation(
    Hb_func, robot.dof, rng, trials=20, vel_scale=1.2, refine=False
)
print(f"选中激励 cond(W) = {cond:.3g}")

data = sample_trajectory(traj, ref.inverse_dynamics, rate=100)
result = identify(Hb_func, data)
err = parameter_error(result.parms, pi_b_true)
print(result.summary())
print(f"相对误差 {err['rel_norm']:.3e}，最大分量误差 {err['abs_max']:.3e}")

# 力矩噪声灵敏度
for std in (0.0, 0.01, 0.05):
    noisy = data if std == 0 else add_noise(data, np.random.default_rng(1), tau_std=std)
    res = identify(Hb_func, noisy)
    e = parameter_error(res.parms, pi_b_true)["rel_norm"]
    print(f"tau_std={std:<4}  参数相对误差 {e:.3e}  残差 RMS {res.residual_rms:.3e}")


## 5. 由基参数计算 M / c / g

单个连杆的质量、质心和惯量通常不能由基参数唯一确定。但对于本例的纯刚体模型，下面三个状态足以分离惯性、速度和重力项：

- `g(q) = Hb(q, 0, 0) pi_b`
- `M(q)[:, i] = Hb(q, 0, e_i) pi_b - g(q)`
- `c(q, dq) = Hb(q, dq, 0) pi_b - g(q)`

这里的小写 `c` 是速度相关力矩向量，不是质心坐标，也不是科氏矩阵 `C`。


In [ ]:
decomposer = DynamicsDecomposer(Hb_func, result.parms, robot.dof)
q_d = np.array([0.4, -0.3, 0.5])
dq_d = np.array([0.2, -0.1, 0.3])
ddq_d = np.array([-0.4, 0.5, -0.2])

terms = decomposer.at(q_d, dq_d)
tau_split = terms.torque(ddq_d)
tau_mj = ref.inverse_dynamics(q_d, dq_d, ddq_d)

print("||tau_split - tau_mj|| =", np.linalg.norm(tau_split - tau_mj))
print("||Hb@pi_hat - tau_mj|| =", np.linalg.norm(Hb_func(q_d, dq_d, ddq_d) @ result.parms - tau_mj))
print("M shape", terms.M.shape, "对称误差", terms.symmetry_error())


无噪声时，参数和力矩误差应接近数值精度；加噪声后的误差还取决于激励轨迹。很小的拟合残差不能单独证明参数准确，需同时检查回归矩阵的秩、条件数和独立轨迹上的预测。

要复现含闭环跟踪、传感器噪声和滤波的实验，请使用 `robot-identify-rrr --realistic`，具体配置和结果见 [README](../../README.md)。
